In [2]:
!pip install optuna

In [3]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

def df_Data_Cleansing(df):
    cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().astype('category')

    df['short_sleep_flag'] = (df['sleep_duration'] < 6.0).astype(int)

    df["exercise_strength"] = df['calorie_expenditure'] / ((df['exercise_duration'] + 0.1) * df['bmi'])

    df['healthy_score'] = (
        df['sleep_duration'].between(7, 8).astype(int) +
        (df['bmi'] <= 19).astype(int) +
        (df['exercise_duration'] >= 40).astype(int) +
        (df['stress_level'] == 'low').astype(int) +
        (df['sleep_quality'] == 'good').astype(int) +
        (df['physical_activity_level'] == 'active').astype(int)
    )

    df['unhealthy_score'] = (
        df['sleep_duration'].between(5, 6).astype(int) +
        (df['bmi'] >= 27).astype(int) +
        (df['calorie_expenditure'] <= 1400).astype(int) +
        (df['step_count'] <= 5000).astype(int) +
        (df['exercise_duration'] <= 30).astype(int) +
        (df['stress_level'] == 'high').astype(int) +
        (df['sleep_quality'] == 'poor').astype(int) +
        df['physical_activity_level'].isin(['sedentary', 'moderate']).astype(int) +
        (df['smoking_alcohol'] == 'yes').astype(int)
    )

    stress_mapping = {'low': 1, 'medium': 2, 'high': 3}
    df['stress_level_num'] = df['stress_level'].map(stress_mapping).astype(float)
    df['stress_per_sleep'] = df['stress_level_num'] / df['sleep_duration']

    drop_cols = ['stress_level_num']
    df = df.drop(columns=drop_cols, errors='ignore')

    return df

# データの読み込み（パスはColabの環境に合わせて変更してください）
df = df_Data_Cleansing(pd.read_csv('/content/train.csv', encoding="utf-8"))

X = df.drop(['id', 'health_condition'], axis=1)
y = df['health_condition']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Optunaの目的関数
def objective(trial):
    # 探索するパラメータの範囲を指定
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_error',
        'verbosity': -1,
        'class_weight': 'balanced',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in cv.split(X, y_encoded):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y_encoded[train_idx], y_encoded[valid_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=1000)

        # Early Stoppingを設定して過学習を防ぐ
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        preds = model.predict(X_valid)
        score = balanced_accuracy_score(y_valid, preds)
        scores.append(score)

    return np.mean(scores)

print("Optunaによるパラメータ探索を開始します...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # まずは30回お試しで探索

print("=========================")
print(f"ベストスコア (Balanced Accuracy): {study.best_value:.5f}")
print("ベストパラメータ:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("=========================")

[I 2026-07-20 03:59:16,424] A new study created in memory with name: no-name-aecdf84c-709b-47d8-8f69-39ce7b0c82be


Optunaによるパラメータ探索を開始します...


[I 2026-07-20 04:01:20,977] Trial 0 finished with value: 0.9488319156537045 and parameters: {'learning_rate': 0.028929068075347925, 'num_leaves': 67, 'max_depth': 10, 'min_child_samples': 76, 'subsample': 0.7677162702374288, 'colsample_bytree': 0.7239052703008524}. Best is trial 0 with value: 0.9488319156537045.
[I 2026-07-20 04:06:07,031] Trial 1 finished with value: 0.9472977836574232 and parameters: {'learning_rate': 0.006363026580260967, 'num_leaves': 26, 'max_depth': 7, 'min_child_samples': 27, 'subsample': 0.7378187631771858, 'colsample_bytree': 0.8561735940863959}. Best is trial 0 with value: 0.9488319156537045.
[I 2026-07-20 04:07:56,078] Trial 2 finished with value: 0.9487478055473086 and parameters: {'learning_rate': 0.010859213181681985, 'num_leaves': 87, 'max_depth': 8, 'min_child_samples': 64, 'subsample': 0.6823558595624063, 'colsample_bytree': 0.8548635531471356}. Best is trial 0 with value: 0.9488319156537045.
[I 2026-07-20 04:09:11,363] Trial 3 finished with value: 0.9

ベストスコア (Balanced Accuracy): 0.94978
ベストパラメータ:
    learning_rate: 0.03644606592256244
    num_leaves: 24
    max_depth: 4
    min_child_samples: 66
    subsample: 0.784240004466416
    colsample_bytree: 0.6914402619376562


# 最適なパラメータ

[I 2026-07-20 08:44:05,184] Trial 22 finished with value: 0.9497803484507361 and parameters: {'learning_rate': 0.03644606592256244, 'num_leaves': 24, 'max_depth': 4, 'min_child_samples': 66, 'subsample': 0.784240004466416, 'colsample_bytree': 0.6914402619376562}. Best is trial 22 with value: 0.9497803484507361.